# **Primetrade.ai Internship Assignment (Data Science)**

---



# Trader Performance vs Market Sentiment

Objective:
Analyze the relationship between Bitcoin market sentiment (Fear/Greed) and trader performance on Hyperliquid.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style("whitegrid")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

Dataset Overview

In [ ]:
print("Fear and Greed Dataset")
print(sentiment.shape)
print()

print("Trade Dataset")
print(trades.shape)

In [ ]:
print(sentiment.columns)
print(trades.columns)

Missing Values

In [ ]:
print("Sentiment Missing Values")
print(sentiment.isnull().sum())
print()

print("Trades Missing Values")
print(trades.isnull().sum())

Duplicate Values

In [ ]:
print("Sentiment Duplicates")
print(sentiment.duplicated().sum())
print()
print("Trades Duplicates")
print(trades.duplicated().sum())

Data Cleaning

In [ ]:
sentiment["data"] = pd.to_datetime(sentiment["date"])

trades["Timestamp IST"] = pd.to_datetime(trades["Timestamp IST"], dayfirst= True)

In [ ]:
trades["date"] = trades["Timestamp IST"].dt.date
trades["date"] = pd.to_datetime(trades["date"])

In [ ]:
sentiment["date"] = pd.to_datetime(sentiment["date"])

trades["date"] = pd.to_datetime(trades["date"])

In [ ]:
print(trades["date"].dtype)
print(sentiment["date"].dtype)

####Merging Datasets

Merged trader data with sentiment data using trading date.

In [ ]:
df = pd.merge(trades,sentiment[["date","classification","value"]], on="date", how="left")

In [ ]:
df.shape

In [ ]:
df.head()

---
## **Feature engineering**
---

After cleaning and merging the datasets, several derived features were created to analyze trader performance and behavior under different market sentiment conditions.

The following metrics were generated:

1. Win/Loss Indicator
2. Daily Profit & Loss (PnL)
3. Trade Count
4. Average Trade Size
5. Long/Short Bias
6. Trader Performance Summary

These features will be used in subsequent analysis and trader segmentation.

####**1. Win/Loss Indicator**

A win/loss indicator was created based on the Closed PnL of each trade.

- Winning Trade → Closed PnL > 0
- Losing Trade → Closed PnL ≤ 0

This feature is later used to calculate trader win rates and identify consistent performers.

In [ ]:
df["win"] = (df["Closed PnL"] > 0).astype(int)

df[["Closed PnL", "win"]].head()

####**2. Daily Profit and Loss (PnL)**

Daily PnL measures the total profit or loss generated by each trader on a given day.

This metric helps evaluate how trader profitability changes under different market sentiment regimes such as Fear and Greed.

In [ ]:
daily_pnl = (df.groupby(["date","Account"])["Closed PnL"].sum().reset_index())
daily_pnl.head()

####**3. Trade Count**

Trade count represents the number of trades executed.

This metric helps distinguish between:

- Frequent Traders
- Infrequent Traders


In [ ]:
trade_count = (df.groupby("Account").size().reset_index(name="trade_count"))

trade_count.head()

In [ ]:
trade_count.describe()

####**4. Average Trade Size**

Average trade size measures the typical amount of capital deployed by a trader.

Larger trade sizes generally imply higher market exposure and potentially higher risk.

In [ ]:
avg_trade_size = (df.groupby("Account")["Size USD"].mean().reset_index())

avg_trade_size.columns = ["Account", "avg_trade_size_usd"]

avg_trade_size.head()

In [ ]:
avg_trade_size.describe()

####**5. Long/Short Bias**

Long/Short Bias examines whether traders preferred long or short positions under different market sentiment conditions.

This metric helps determine whether trader positioning changes during Fear and Greed periods.

In [ ]:
direction_analysis = pd.crosstab(df["classification"], df["Direction"], normalize="index")

direction_analysis

####**6. Trading Activity by Market Sentiment**

To understand behavioral differences, trading activity was analyzed separately for each market sentiment category.

The total number of trades executed under Fear and Greed conditions was calculated.

In [ ]:
sentiment_trade_count = (df.groupby("classification").size().reset_index(name="trade_count"))

sentiment_trade_count

####**Trader Performance Summary**

A consolidated trader-level summary was created by combining:

- Average Profitability
- Win Rate
- Average Trade Size
- Trade Frequency

This summary is later used for trader segmentation and clustering analysis.

In [ ]:
account_summary = (df.groupby("Account").agg({"Closed PnL": "mean", "win": "mean", "Size USD": "mean", "Trade ID": "count"}).reset_index())

account_summary.columns = ["Account", "avg_pnl", "win_rate", "avg_trade_size", "trade_count"]

account_summary.head()

In [ ]:
account_summary.describe()

###*Feature Engineering Summary*

The following derived features were successfully created:

| Feature | Description |
|----------|-------------|
| win | Indicates whether a trade was profitable |
| daily_pnl | Daily profit/loss per trader |
| trade_count | Number of trades executed |
| avg_trade_size | Average capital deployed per trade |
| direction_analysis | Long vs Short preference |
| account_summary | Consolidated trader performance metrics |

These engineered features form the foundation for the Fear vs Greed analysis, trader segmentation, and strategy recommendation sections that follow.

---
##**Exploratory Data Analysis**
---
We investigate how trader performance and behavior vary across different market sentiment conditions.

The analysis focuses on:

1. Profitability (PnL)
2. Win Rate
3. Trade Size
4. Trading Activity
5. Long/Short Bias

The objective is to determine whether Fear and Greed influence trading outcomes and trader behavior.

###**1. Market Sentiment Distribution**

Before comparing trader performance, it is useful to understand the distribution of sentiment categories in the dataset.

In [ ]:
sentiment_counts = (df["classification"].value_counts())

sentiment_counts

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(data=df, x="classification", order=df["classification"].value_counts().index)

plt.title("Market Sentiment Distribution")
plt.xticks(rotation=45)

plt.show()

###**2. Profitability Across Sentiment Regimes**

Profitability is measured using Closed PnL.

The objective is to determine whether traders perform differently during Fear and Greed periods.

In [ ]:
pnl_analysis = (df.groupby("classification")["Closed PnL"].agg(["count", "mean", "median", "sum"]))

pnl_analysis

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(data=df, x="classification", y="Closed PnL")

plt.xticks(rotation=45)

plt.title("PnL Distribution by Sentiment")

plt.show()

### Observation

The boxplot highlights differences in trader profitability across sentiment conditions.

Higher average and median PnL values suggest stronger performance under specific market sentiment regimes.

###**3. Win Rate Analysis**

Win rate measures the proportion of profitable trades.

This metric helps evaluate whether sentiment affects trading success rates.

In [ ]:
win_rate_analysis = (df.groupby("classification")["win"].mean()* 100)

win_rate_analysis

In [ ]:
win_rate_analysis.sort_values().plot(kind="bar", figsize=(10,5))

plt.ylabel("Win Rate (%)")
plt.title("Win Rate by Sentiment")

plt.show()

###**4. Average Trade Size**

Average trade size measures how much capital traders deploy under different market conditions.

Larger position sizes may indicate greater confidence or risk-taking behavior.

In [ ]:
trade_size_analysis = (df.groupby("classification")["Size USD"].mean())

trade_size_analysis

In [ ]:
trade_size_analysis.plot(kind="bar", figsize=(10,5))

plt.ylabel("Average Trade Size (USD)")
plt.title("Average Trade Size by Sentiment")

plt.show()

###**5. Trading Activity**

Trading activity is measured using the total number of trades executed under each sentiment condition.

Higher activity levels may indicate stronger trader participation.

In [ ]:
activity_analysis = (df.groupby("classification").size().reset_index(name="trade_count"))

activity_analysis

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    data=activity_analysis,
    x="classification",
    y="trade_count"
)

plt.xticks(rotation=45)

plt.title("Trade Count by Sentiment")

plt.show()

###**6. Long vs Short Bias**

This analysis investigates whether traders prefer long or short positions under different sentiment conditions.

In [ ]:
direction_analysis = pd.crosstab(df["classification"], df["Direction"], normalize="index")

direction_analysis

In [ ]:
direction_analysis.plot(kind="bar", stacked=True, figsize=(10,6))

plt.title("Long vs Short Bias by Sentiment")

plt.ylabel("Percentage")

plt.show()

---
## **Statistical Significance Test**

---

A Welch's t-test was performed to determine whether profitability differs significantly between Fear and Greed periods.

###*Interpretation:*

If p-value < 0.05:

The difference in profitability between Fear and Greed periods is statistically significant.

If p-value ≥ 0.05:

The observed difference may be due to random variation.

In [ ]:
fear = df[df["classification"].str.contains("Fear",na=False)]["Closed PnL"]

greed = df[df["classification"].str.contains("Greed",na=False)]["Closed PnL"]

In [ ]:
t_stat, p_value = ttest_ind(fear, greed, equal_var=False)

print("T-stat:", t_stat)
print("P-value:", p_value)

Since the p-value is greater than the 0.05 significance threshold, the difference in average profitability between Fear and Greed periods is not statistically significant.

This suggests that market sentiment alone may not be a strong predictor of trader profitability. Other factors such as trading frequency, position sizing, trader experience, and risk management practices may have a greater influence on trading outcomes.

---
#**Trader Segmentation**
---

To better understand trader behavior, traders were segmented based on their activity levels and trading performance.

The objective is to identify distinct trader groups and compare their profitability characteristics.

The following segments were created:

1. Frequent vs Infrequent Traders
2. Consistent Winners vs Inconsistent Traders
3. Behavioral Clusters using K-Means Clustering

Account Metrics

In [ ]:
account_stats = (df.groupby("Account").agg({"Closed PnL":"mean", "win":"mean", "Size USD":"mean", "Trade ID":"count"}))

In [ ]:
account_stats.columns = ["avg_pnl", "win_rate", "avg_trade_size", "trade_count"]

## **Frequent vs Infrequent**

Traders were divided into two groups based on the median number of trades executed.

- Frequent Traders: Above median trade count
- Infrequent Traders: Below median trade count

This segmentation helps determine whether higher trading activity is associated with improved profitability.

In [ ]:
median_trade_count = (account_stats["trade_count"].median())

account_stats["frequency_segment"] = np.where(account_stats["trade_count"] > median_trade_count, "Frequent", "Infrequent")

In [ ]:
frequency_results = (account_stats.groupby("frequency_segment")[["avg_pnl","win_rate","trade_count"]].mean())

frequency_results

## **Consistent Winners**

Traders with win rates greater than 55% were classified as Consistent Winners.

All remaining traders were classified as Inconsistent Traders.

This analysis investigates whether consistently successful traders exhibit different trading behaviors.

In [ ]:
account_stats["winner_segment"] = np.where(account_stats["win_rate"] > 0.55,"Consistent Winner", "Inconsistent")

In [ ]:
winner_results = (account_stats.groupby("winner_segment")[["avg_pnl","win_rate","avg_trade_size"]].mean())

winner_results


## **Clustering**



**K-Means** clustering was used to identify naturally occurring trader groups based on:

- Average Profitability
- Win Rate
- Average Trade Size
- Trading Frequency

This approach helps uncover trader archetypes without manually defining categories.

In [ ]:
cluster_features = account_stats[["avg_pnl", "win_rate", "avg_trade_size", "trade_count"]]

In [ ]:
scaler = StandardScaler()

X = scaler.fit_transform(cluster_features)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)

account_stats["Cluster"] = kmeans.fit_predict(X)

In [ ]:
account_stats.groupby("Cluster").mean(
    numeric_only=True
)

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=account_stats,
    x="trade_count",
    y="avg_pnl",
    hue="Cluster"
)

plt.title("Trader Clusters")

plt.xlabel("Trade Count")
plt.ylabel("Average PnL")

plt.savefig("cluster_plot.png", bbox_inches="tight")

plt.show()

---
#**Key Insights**
---

## 1: Extreme Greed Produced the Best Trading Outcomes

Extreme Greed periods generated the highest average trader profitability (67.89) and the highest win rate (46.49%).

This suggests traders generally benefited from strong directional market momentum during highly optimistic market conditions.



## 2: Traders Took Larger Risks During Fear

Despite lower profitability, traders deployed the largest average position sizes during Fear periods (approximately $7,816 per trade).

This indicates that traders increased capital exposure during uncertain market conditions, potentially increasing downside risk.



## 3: Fear Generated the Highest Trading Activity

Fear periods accounted for the highest number of trades (61,837 trades).

This suggests traders were more active during uncertain markets, likely attempting to capture volatility-driven opportunities.



## 4: Trading More Frequently Did Not Necessarily Improve Profitability

Frequent traders achieved a slightly higher win rate (41.36%) but lower average profitability (57.58).

In contrast, infrequent traders generated substantially higher average profitability (137.80) despite lower trade frequency.

This suggests that trade quality may be more important than trade quantity.



## 5: Consistent Winners Used Smaller Position Sizes

Consistent winners achieved a win rate of 69.2% while using significantly smaller average position sizes ($2,334).

Inconsistent traders deployed much larger average positions ($6,253) but achieved lower win rates (38.4%).

This suggests disciplined position sizing may contribute to long-term trading success.



## 6: Market Sentiment Alone Was Not Statistically Significant

Although profitability appeared higher during Greed conditions, the Welch's t-test produced a p-value of 0.323.

Therefore, the difference in profitability between Fear and Greed periods was not statistically significant at the 5% significance level.

This indicates that trader behavior and risk management may be more important than sentiment alone.

---
# **Strategy Recommendations**
---

Based on the analysis, the following trading guidelines are proposed.

## Strategy 1: Reduce Position Sizes During Fear Periods

Observation:

Traders used the largest average position sizes during Fear periods while achieving lower profitability compared to Extreme Greed periods.

Recommendation:

- Reduce leverage and position sizes during Fear conditions.
- Focus on capital preservation rather than aggressive risk-taking.
- Apply stricter stop-loss levels.

Expected Benefit:

Lower drawdowns during volatile market conditions.
## Strategy 2: Prioritize High-Conviction Trades Over High Frequency

Observation:

Infrequent traders generated substantially higher average profitability than frequent traders.

Recommendation:

- Avoid overtrading.
- Enter positions only when strong setups are present.
- Use sentiment as a confirmation signal rather than a primary trading trigger.

Expected Benefit:

Improved risk-adjusted returns and reduced transaction costs.
## Strategy 3: Adopt Position Sizing Patterns of Consistent Winners

Observation:

Consistent winners maintained much smaller average position sizes while achieving significantly higher win rates.

Recommendation:

- Limit position size as a percentage of account capital.
- Scale positions gradually instead of taking large exposures.
- Emphasize consistency over short-term gains.

Expected Benefit:

More stable profitability and improved long-term survival.